In [ ]:
import os
os.chdir('..')

In [ ]:
import pandas as pd

# Load train and test datasets from the project's raw_data folder
train = pd.read_csv('raw_data/train.csv')
test = pd.read_csv('raw_data/test.csv')

df = train.copy() 

# Check the dimensions of both datasets (rows, columns)
print(train.shape)
print(test.shape)

# Preview the first 5 rows of the training set
train.head()


In [ ]:
import os
print(os.getcwd())
print(os.listdir('.'))  # see what's actually here

In [ ]:
# Check column names and data types
print(train.dtypes)

In [ ]:
# Check for missing values in each column
train.isnull().sum().sort_values(ascending=False)

In [ ]:
# Summary statistics for all numeric columns
train.describe()

In [ ]:
import matplotlib.pyplot as plt

# Distribution of our target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(train['actual_finish_time_minutes'].dropna(), bins=50, edgecolor='white', color='steelblue')
axes[0].set_title('Finish Time Distribution')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Count')

# Boxplot to spot outliers
axes[1].boxplot(train['actual_finish_time_minutes'].dropna(), vert=False)
axes[1].set_title('Finish Time Boxplot')
axes[1].set_xlabel('Minutes')

plt.tight_layout()
plt.show()

# Key stats
print(train['actual_finish_time_minutes'].describe())


In [ ]:
# Check all categorical columns and their unique values
cat_cols = train.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f"\n{col} ({train[col].nunique()} unique):")
    print(train[col].value_counts())
    

In [ ]:
# Correlation of all numeric features with the target
corr = (train
        .select_dtypes(include='number')
        .corr()['actual_finish_time_minutes']
        .drop('actual_finish_time_minutes')
        .sort_values())

# Plot as horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 10))
colors = ['steelblue' if v < 0 else 'tomato' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Finish Time', fontsize=14)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

# Print top 5 positive and negative correlates
print("🔴 Top 5 features that INCREASE finish time (slower):")
print(corr.tail(5))
print("\n🔵 Top 5 features that DECREASE finish time (faster):")
print(corr.head(5))

In [ ]:
# Boxplots of finish time by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Finish time by training program (ordinal)
order = ['Beginner', 'Intermediate', 'Advanced']
for i, (col, ax) in enumerate(zip(
    ['training_program', 'gender', 'marathon_weather', 'course_difficulty'],
    axes.flatten()
)):
    groups = [train[train[col] == cat]['actual_finish_time_minutes'].dropna()
              for cat in train[col].unique()]
    labels = train[col].unique()
    ax.boxplot(groups, labels=labels, vert=True)
    ax.set_title(f'Finish Time by {col}')
    ax.set_ylabel('Minutes')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots of the strongest numeric predictors vs finish time
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pairs = [
    ('running_experience_months', 'Experience (months)'),
    ('speed_work_sessions_per_week', 'Speed Sessions/Week'),
    ('runs_per_week', 'Runs Per Week'),
    ('rest_days_per_week', 'Rest Days/Week'),
    ('resting_heart_rate_bpm', 'Resting Heart Rate'),
    ('vo2_max', 'VO2 Max'),
]

for ax, (col, label) in zip(axes.flatten(), pairs):
    ax.scatter(train[col], train['actual_finish_time_minutes'],
               alpha=0.1, s=5, color='steelblue')
    ax.set_xlabel(label)
    ax.set_ylabel('Finish Time (min)')
    ax.set_title(f'{label} vs Finish Time')

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

# Pairplot of top numeric predictors
cols = ['running_experience_months', 'speed_work_sessions_per_week',
        'runs_per_week', 'vo2_max', 'resting_heart_rate_bpm', 'actual_finish_time_minutes']

sns.pairplot(train[cols].dropna(), plot_kws={'alpha': 0.1, 's': 5})
plt.show()

In [ ]:
# Does gender interact with training program?
sns.boxplot(data=train, x='training_program', y='actual_finish_time_minutes',
            hue='gender', order=['Beginner', 'Intermediate', 'Advanced'])
plt.title('Finish Time by Training Program and Gender')
plt.show()

In [ ]:
# Profile of runners who DNF'd vs finished
train['dnf'] = train['actual_finish_time_minutes'].isna().astype(int)

print(train.groupby('dnf')[['age', 'running_experience_months',
      'injury_count', 'weekly_mileage_km','sleep_hours_avg']].mean())

In [ ]:
# Extract month and look at average finish time per month
train['month'] = pd.to_datetime(train['marathon_date']).dt.month

train.groupby('month')['actual_finish_time_minutes'].mean().plot(
    kind='bar', color='steelblue', edgecolor='white')
plt.title('Average Finish Time by Race Month')
plt.xlabel('Month')
plt.ylabel('Avg Finish Time (min)')
plt.show()

In [ ]:
# Fill NaN as 'None' temporarily and compare finish times
train['injury_severity_filled'] = train['injury_severity'].fillna('None')

order = ['None', 'Minor', 'Moderate', 'Severe']
sns.boxplot(data=train, x='injury_severity_filled',
            y='actual_finish_time_minutes', order=order)
plt.title('Finish Time by Injury Severity')
plt.show()


In [ ]:
# ============================================================
# EDA FINDINGS SUMMARY — based on confirmed outputs
# ============================================================

findings = """
=== DATASET OVERVIEW ===
- 80,000 runners in train, 20,000 in test, 41 features
- Target: actual_finish_time_minutes (range 171–400 min, mean ~288 min / 4h48)
- 1,989 missing actual_finish_time_minutes → DNFs (did not finish)

=== MISSING VALUES ===
- injury_severity: 47,350 missing (59%) → NaN means no injury, recode as 'None'
- personal_best_minutes: 10,745 missing → likely first-time marathoners
- vo2_max: 9,611 missing → not everyone gets lab tested
- cross_training, nutrition_score, hydration_consistency: moderate missingness (self-reported)
- sleep_hours_avg: 3,942 missing

=== CATEGORICAL FEATURES ===
- gender: Male ~48% / Female ~48% / Non-binary ~4%
  → Weak predictor — almost no difference in finish time across genders
- training_program: Beginner (50%) / Intermediate (32%) / Advanced (18%)
  → Strong ordinal predictor: Beginner ~310 / Intermediate ~280 / Advanced ~245 min
- marathon_weather: 6 categories, fairly balanced
  → Hot weather shows slightly higher finish times, minimal difference overall
- course_difficulty: Flat / Mixed / Hilly fairly balanced
  → Hilly courses show noticeably higher finish times than Flat
- marathon_date: finish time flat across all 12 months (~288 min)
  → Month/season is NOT a useful feature → drop it

=== NUMERIC FEATURES ===
- running_experience_months: strongest legitimate predictor (r=-0.65)
  → Strong non-linear negative relationship, most gains from 0–50 months
- speed_work_sessions_per_week (r=-0.35): more sessions = faster, clear step pattern
- runs_per_week (r=-0.31): more runs = faster, clear step pattern
- rest_days_per_week (r=+0.30): more rest = slower
- resting_heart_rate_bpm (r=+0.20): higher HR = slightly slower, noisy
- vo2_max (r=-0.35): higher VO2 max = meaningfully faster

=== INJURY SEVERITY ===
- Clear ordinal effect: None ~285 / Minor ~293 / Moderate ~305 / Severe ~310 min
- Encode ordinally as: None=0, Minor=1, Moderate=2, Severe=3

=== DNF ANALYSIS ===
- DNF runners are older (+2.4 yrs), less experienced (-8 months), more injuries (1.12 vs 0.54)
- Implication: drop DNFs from training set or model separately

=== LEAKY FEATURES — exclude from model ===
- target_finish_time_minutes (r=0.95) — runner's own pre-race goal
- personal_best_minutes (r=0.86) — use carefully, may leak
- medal_outcome (r=-0.38) — race outcome, not a predictor
- weekly_mileage_miles — exact duplicate of weekly_mileage_km

=== PREPROCESSING PLAN ===
1. Recode injury_severity NaN → 'None', encode ordinally (0–3)
2. Flag first-time marathoners where personal_best_minutes is NaN
3. Encode training_program ordinally (Beginner=0, Intermediate=1, Advanced=2)
4. OHE: marathon_weather, course_difficulty, gender
5. Drop: marathon_date, weekly_mileage_miles, target_finish_time_minutes, medal_outcome
6. Drop DNF rows (actual_finish_time_minutes is NaN)
7. Build baseline model (Linear Regression)
"""

print(findings)

In [ ]:
# Copy
train_clean = train.copy()
test_clean = test.copy()

In [ ]:
# Step 1 — Drop DNF rows (no finish time = can't train on them)
print(f"Before: {train_clean.shape}")
train_clean = train_clean.dropna(subset=['actual_finish_time_minutes'])
print(f"After dropping DNFs: {train_clean.shape}")

In [ ]:
# Check remaining missing values after dropping DNFs
train_clean.isnull().sum().sort_values(ascending=False)

In [ ]:
# Full list of features in the dataset
print(list(train_clean.columns))


In [ ]:
# Full list of features in a readable format
for i, col in enumerate(train_clean.columns, 1):
    print(f"{i:02d}. {col}")
    

In [ ]:
!git add .
!git commit -m "My Notebook"
!git push origin HEAD:Francesco

In [ ]:
training_map = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
course_map = {"Flat": 1, "Mixed": 2, "Hilly": 3}
injury_map = {"Minor": 1, "Moderate": 2, "Severe": 3}

In [ ]:
# Encode on train_clean
train_clean['training_program_enc'] = train_clean['training_program'].map(training_map)
train_clean['course_difficulty_enc'] = train_clean['course_difficulty'].map(course_map)
train_clean['injury_severity_enc'] = train_clean['injury_severity'].map(injury_map).fillna(0)
train_clean = pd.get_dummies(train_clean, columns=['marathon_weather'], drop_first=True)

# Handle missing values
train_clean['personal_best_flag'] = train_clean['personal_best_minutes'].isna().astype(int)
train_clean['personal_best_minutes'] = train_clean['personal_best_minutes'].fillna(train_clean['personal_best_minutes'].median())
train_clean['vo2_max'] = train_clean['vo2_max'].fillna(train_clean['vo2_max'].median())

# Drop unneeded columns
cols_to_drop = ['target_finish_time_minutes', 'medal_outcome', 'weekly_mileage_miles',
                'marathon_date', 'month', 'injury_severity_filled', 'dnf',
                'training_program', 'course_difficulty', 'injury_severity']
train_clean = train_clean.drop(columns=[c for c in cols_to_drop if c in train_clean.columns])

# Create df
df = train_clean.copy()

In [ ]:
# Create age buckets based on marathon age categories
bins = [18, 35, 45, 55, 100]
labels = ['18-34', '35-44', '45-54', '55+']
train_clean['age_bucket'] = pd.cut(train_clean['age'], bins=bins, labels=labels, right=False)

# Correlation of numeric features with finish time, split by age bucket
numeric_cols = train_clean.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'actual_finish_time_minutes']

corr_by_age = (train_clean
               .groupby('age_bucket', observed=True)[numeric_cols + ['actual_finish_time_minutes']]
               .apply(lambda g: g.corr()['actual_finish_time_minutes'].drop('actual_finish_time_minutes'))
)

# Plot heatmap
fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(corr_by_age, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax)
ax.set_title('Correlation with Finish Time by Age Bucket')
ax.set_xlabel('Feature')
ax.set_ylabel('Age Bucket')
plt.tight_layout()
plt.show()


In [ ]:
cols_to_drop = ['dnf', 'weekly_mileage_miles', 'medal_outcome', 
                'target_finish_time_minutes', 'month', 'injury_severity_filled']

train_clean = train_clean.drop(columns=[c for c in cols_to_drop if c in train_clean.columns])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

bins = [18, 35, 45, 55, 100]
labels = ['18-34', '35-44', '45-54', '55+']
train_clean['age_bucket'] = pd.cut(train_clean['age'], bins=bins, labels=labels, right=False)

for ax, bucket in zip(axes.flatten(), labels):
    subset = train_clean[train_clean['age_bucket'] == bucket]
    
    corr = (subset
            .select_dtypes(include='number')
            .corr()['actual_finish_time_minutes']
            .drop(['actual_finish_time_minutes', 'age'])
            .sort_values())
    
    colors = ['steelblue' if v < 0 else 'tomato' for v in corr]
    corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Feature Correlation with Finish Time — {bucket}', fontsize=12)
    ax.set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# EDA FINDINGS — CORRELATION ANALYSIS BY SUBGROUP
# ============================================================

## BY AGE GROUP
# ------------------------------------------------------------
# CONSISTENT ACROSS ALL AGES:
# - running_experience_months → strongest negative predictor in every group
# - training_program_enc → strong negative predictor, stable across all ages
# - personal_best_minutes → strong positive (semi-leaky, use carefully)
# - rest_days_per_week → positive across all groups (more rest = slower)
#
# CHANGES WITH AGE:
# - resting_heart_rate_bpm: grows as a predictor with age
#   → worth adding interaction term: age × resting_heart_rate_bpm
# - injury_severity_enc + injury_count: increasingly important for older runners
#   → worth adding interaction term: age × injury_count
# - running_experience_months: weakens for 55+
#   → experience matters less when physical decline dominates
# - vo2_max: consistent but slightly weaker in 55+
#   → natural VO2 max decline reduces variance in older group
#
# SUGGESTED FEATURES BY AGE:
# 18-34 → running_experience_months, vo2_max, speed_work_sessions_per_week,
#          training_program_enc, runs_per_week
# 35-44 → same as 18-34 + resting_heart_rate_bpm starts to matter
# 45-54 → resting_heart_rate_bpm, injury_count, vo2_max,
#          training_program_enc, missed_workout_pct
# 55+   → injury_severity_enc, injury_count, resting_heart_rate_bpm,
#          training_program_enc, course_difficulty_enc

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, program in zip(axes.flatten(), ['Beginner', 'Intermediate', 'Advanced']):
    subset = train_clean[train_clean['training_program_enc'] == training_map[program]]
    
    corr = (subset
            .select_dtypes(include='number')
            .corr()['actual_finish_time_minutes']
            .drop(['actual_finish_time_minutes'])
            .sort_values())
    
    colors = ['steelblue' if v < 0 else 'tomato' for v in corr]
    corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Feature Correlation — {program}', fontsize=12)
    ax.set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()

## BY TRAINING PROGRAM
# ------------------------------------------------------------
# BEGINNER:
# - vo2_max is the dominant negative predictor
#   → raw fitness is what separates runners when training is basic
# - running_experience_months has almost no effect
#   → experience without proper training doesn't translate to speed
#
# INTERMEDIATE:
# - More balanced: vo2_max, running_experience_months, speed_work_sessions_per_week
#   all contribute meaningfully
# - Transition zone — both fitness and training quality matter
#
# ADVANCED:
# - speed_work_sessions_per_week and running_experience_months dominate
# - vo2_max weakens — structured training compensates for lower aerobic capacity
#   → at elite level, training discipline > raw fitness
#
# SUGGESTED FEATURES BY TRAINING LEVEL:
# Beginner     → vo2_max, injury_count, missed_workout_pct, course_difficulty_enc
# Intermediate → vo2_max, running_experience_months, speed_work_sessions_per_week,
#                training_streak_days
# Advanced     → speed_work_sessions_per_week, running_experience_months,
#                long_run_distance_km, training_streak_days
#
# INTERACTION TERMS TO CONSIDER:
# → training_program_enc × vo2_max
# → training_program_enc × running_experience_months
# → training_program_enc × speed_work_sessions_per_week

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, gender in zip(axes.flatten(), train_clean['gender'].unique()):
    subset = train_clean[train_clean['gender'] == gender]
    
    corr = (subset
            .select_dtypes(include='number')
            .corr()['actual_finish_time_minutes']
            .drop(['actual_finish_time_minutes'])
            .sort_values())
    
    colors = ['steelblue' if v < 0 else 'tomato' for v in corr]
    corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Feature Correlation — {gender}', fontsize=12)
    ax.set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()

## BY GENDER
# ------------------------------------------------------------
# - Patterns are almost identical across Male, Female and Non-binary
# - running_experience_months and training_program_enc are top negative predictors
#   for all three groups
# - Non-binary group shows noisier correlations due to small sample size (~4%)
#
# CONCLUSION:
# → Gender is NOT a useful predictor — drop it from the model
# → Do NOT create gender interaction terms, no signal to exploit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

senior = train_clean[train_clean['age_bucket'] == '55+']

for ax, program in zip(axes.flatten(), ['Beginner', 'Intermediate', 'Advanced']):
    subset = senior[senior['training_program_enc'] == training_map[program]]
    
    corr = (subset
            .select_dtypes(include='number')
            .corr()['actual_finish_time_minutes']
            .drop(['actual_finish_time_minutes'])
            .sort_values())
    
    colors = ['steelblue' if v < 0 else 'tomato' for v in corr]
    corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'55+ Runners — {program} Program', fontsize=12)
    ax.set_xlabel('Pearson Correlation')

plt.suptitle('Feature Correlation with Finish Time — 55+ Age Group by Training Program', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 55+ BY TRAINING PROGRAM
# ------------------------------------------------------------
# BEGINNER 55+:
# - vo2_max strongest negative predictor — raw fitness is everything
# - running_experience_months almost zero — experience alone doesn't help
# - injury_severity_enc + injury_count very strong positive predictors
#   → undertrained senior runners are extremely vulnerable to injuries
#
# INTERMEDIATE 55+:
# - More balanced: vo2_max, speed_work_sessions_per_week,
#   running_experience_months all contribute
# - hydration_consistency appears as a notable negative predictor
#   → unique to this segment, worth exploring as a feature for 55+ models
#
# ADVANCED 55+:
# - running_experience_months finally becomes meaningful
#   → experience pays off for senior runners only when combined with serious training
# - injury_severity_enc remains the top positive predictor even here
#   → injuries are the single biggest drag for 55+ regardless of training level
# - course_difficulty_enc more relevant than in younger advanced runners
#   → senior advanced runners are more sensitive to hilly courses
# - vo2_max weakens vs Beginner 55+
#   → structured training compensates for aerobic decline
#
# KEY INSIGHT FOR 55+ MODELLING:
# → Injuries dominate regardless of training level — always include injury features
# → The path to being faster shifts: fitness (Beginner) → experience+training (Advanced)
#
# SUGGESTED INTERACTION TERMS FOR 55+:
# → age × injury_count
# → age × resting_heart_rate_bpm
# → training_program_enc × injury_severity_enc  (especially powerful for 55+)
# → training_program_enc × vo2_max

## OVERALL FEATURE SELECTION RECOMMENDATION
# ------------------------------------------------------------
# CORE FEATURES (use for all runners):
# - running_experience_months
# - training_program_enc
# - vo2_max
# - speed_work_sessions_per_week
# - runs_per_week
# - resting_heart_rate_bpm
# - injury_severity_enc
# - injury_count
# - course_difficulty_enc
# - missed_workout_pct
# - personal_best_minutes (use carefully — semi-leaky)
# - personal_best_flag
#
# INTERACTION TERMS TO ENGINEER:
# - age × injury_count
# - age × resting_heart_rate_bpm
# - training_program_enc × vo2_max
# - training_program_enc × running_experience_months
# - training_program_enc × injury_severity_enc
#
# DROP:
# - gender (no signal)
# - marathon_weather (minimal signal)
# - target_finish_time_minutes (leaky)
# - medal_outcome (leaky)
# - weekly_mileage_miles (duplicate)
# - marathon_date / month (no signal)

In [ ]:
bins = [18, 35, 45, 55, 100]
labels = ['18-34', '35-44', '45-54', '55+']
train_clean['age_bucket'] = pd.cut(train_clean['age'], bins=bins, labels=labels, right=False)

df = train_clean.copy()

In [ ]:
X_55 = df_55.drop(columns=['actual_finish_time_minutes', 'age_bucket', 'gender', 'runner_id'])

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Filter to 55+ runners only
df_55 = df[df['age_bucket'] == '55+'].copy()

# Keep only numeric columns and drop target
X_55 = df_55.select_dtypes(include='number').drop(columns=['actual_finish_time_minutes'])
y_55 = df_55['actual_finish_time_minutes']

# Fill NaNs with median
X_55 = X_55.fillna(X_55.median())

# Cast to float32 to avoid overflow in matrix operations
X_55 = X_55.astype(np.float32)
y_55 = y_55.astype(np.float32)

# Pipeline: scale + Ridge with explicit solver
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0, solver='cholesky'))
])

scores = cross_val_score(pipeline, X_55, y_55, cv=5, scoring='neg_root_mean_squared_error')

print(f"55+ Ridge Regression Baseline")
print(f"Features used: {X_55.shape[1]}")
print(f"RMSE per fold: {[-round(s, 2) for s in scores]}")
print(f"Mean RMSE:     {-scores.mean():.2f} min")
print(f"Std RMSE:      {scores.std():.2f} min")

In [ ]:
import numpy as np

# Check for inf values
print("Inf values per column:")
print(X_55.columns[np.isinf(X_55).any()].tolist())

# Check for extreme values
print("\nMax absolute value per column:")
print(X_55.abs().max().sort_values(ascending=False).head(10))

# Check actual values after scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_55)
print("\nMax value after scaling:", np.abs(X_scaled).max())
print("Any inf after scaling:", np.isinf(X_scaled).any())
print("Any nan after scaling:", np.isnan(X_scaled).any())